# Build the project dataset

Loads IRS 990 filings (already ingested), FEC Super PAC totals, LDA lobbying filings, and 118th Congress bills into `data/irs990_full.db`. Run top-to-bottom.

**Expected runtimes:**
- IRS: already done (4,472,546 filings across 2019-2026, parser v3)
- FEC committee totals: ~1 min
- LDA 2023+2024: ~70 min with `LDA_API_KEY` in `.env`, ~8 hrs without
- Congress bills: ~5 min (16,565 bills, stub mode)


In [ ]:
from pathlib import Path
import sqlite3
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
DB_PATH = ROOT / 'data' / 'irs990_full.db'

from extract.db import init_db
init_db(DB_PATH)

def row_count(table):
    with sqlite3.connect(DB_PATH) as c:
        return c.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]

print(f"IRS filings:        {row_count('irs990_filings'):>10,}")
print(f"FEC committees:     {row_count('committees'):>10,}")
print(f"FEC disbursements:  {row_count('fec_disbursements'):>10,}")
print(f"LDA filings:        {row_count('lda_filings'):>10,}")
print(f"LDA activities:     {row_count('lda_lobbying_activities'):>10,}")
print(f"Bills:              {row_count('bills'):>10,}")


## IRS 990

4,472,546 filings loaded (submission years 2019-2026, parser irs990-v3). Uncomment the lines below only if re-ingesting from scratch.


In [ ]:
# Already complete - skip unless re-ingesting from scratch (parser irs990-v3)
# from extract.irs990 import ingest_990_directory, ingest_990_zip_folder
# ingest_990_directory(ROOT / 'drive' / '2021Redo_allCycles', db_path=DB_PATH)
# ingest_990_directory(ROOT / 'drive' / '2022Redo_cycle01_41', db_path=DB_PATH)
# for year in [2019, 2020, 2023, 2024, 2025, 2026]:
#     ingest_990_zip_folder(ROOT / 'drive' / 'IRS Form 990 XML Data' / 'XML Zips' / str(year), db_path=DB_PATH)
print(f"IRS filings in db: {row_count('irs990_filings'):,}")


## LDA lobbying filings

Pulls 2023 and 2024 filings. Uses 8 concurrent workers with a shared token-bucket rate limiter (110 req/min). Re-runs are safe via upsert on `filing_uuid`.


In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', datefmt='%H:%M:%S')

from extract.lda import LdaCollector

lda = LdaCollector(db_path=DB_PATH, workers=8, requests_per_minute=110)
for year in [2023, 2024]:
    before = row_count('lda_filings')
    n = lda.collect_filings(filing_year=year)
    print(f"{year}: {n:,} fetched, {row_count('lda_filings') - before:,} new rows")

print(f"\nTotal LDA filings:    {row_count('lda_filings'):,}")
print(f"Total LDA activities: {row_count('lda_lobbying_activities'):,}")


## FEC Super PAC committee totals

Pulls cycle-level summaries for ~2,062 active Super PACs via the `totals/pac-party` endpoint. Takes about 1 minute. For per-committee Schedule B detail after entity matching, use `fec.collect_disbursements_for_committee(committee_id, cycle)`.


In [ ]:
from extract.fec import FecCollector

fec = FecCollector(db_path=DB_PATH)
before_comm = row_count('committees')
before_disb = row_count('fec_disbursements')
n = fec.collect_committee_totals(cycle=2024, committee_type='O')
print(f"Super PACs (O): {n:,} committee totals")
print(f"  committees new:        {row_count('committees') - before_comm:,}")
print(f"  fec_disbursements new: {row_count('fec_disbursements') - before_disb:,} (summary rows)")
print(f"\nTotal committees:        {row_count('committees'):,}")
print(f"Total fec_disbursements: {row_count('fec_disbursements'):,}")


## 118th Congress bills

Pulls HR, S, HJRes, and SJRes bills in stub mode: one paginated request per 250 bills. 16,565 bills in about 5 minutes vs. 16+ hours with per-bill detail requests. For specific bills found via LDA matching, call `congress.collect_bill_detail(bill_id)` for actions, sponsors, and text version URLs.


In [ ]:
from extract.congress import CongressCollector

congress = CongressCollector(db_path=DB_PATH)
for bill_type in ['hr', 's', 'hjres', 'sjres']:
    before = row_count('bills')
    n = congress.collect_bills(congress=118, bill_type=bill_type)
    print(f"Bills ({bill_type:5}): {n:,} fetched, {row_count('bills') - before:,} new")

print(f"\nTotal bills: {row_count('bills'):,}")


## Prepare analysis layers

Syncs org names into the entity observation table, generates name-match candidates, extracts bill references from LDA activity text, and rebuilds the analysis views.


In [ ]:
from extract.pipeline import refresh_analysis_layers

result = refresh_analysis_layers(DB_PATH)
print(result.as_dict())
